In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#cell1 %% [code]
import subprocess, sys, os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(f"找不到 {pattern}，请检查是否在右侧 Add Data 挂载了对应的依赖库！")

# 100% 照抄原代码：寻找并安装 onnxruntime

ONNX_WHL = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")
if ONNX_WHL.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
    print("ONNX Runtime installed")
else:
    print('没有！')



# 100% 照抄原代码：寻找并降级/升级 TF 2.20 以解决死锁
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)
    print("✅ TF 2.20 installed from wheel")
except Exception as e:
    print(e)

# 导入所有后续需要的包
import onnxruntime as ort
import pandas as pd
import numpy as np
import librosa
from tqdm.auto import tqdm
import re


BASE = Path("/kaggle/input/competitions/birdclef-2026")
TRAIN_AUDIO_DIR = BASE / "train_audio"
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")

SR = 32000
WINDOW_SEC = 5
HOP_SEC = 2.5
WINDOW_SAMPLES = int(SR * WINDOW_SEC)   # 160000
HOP_SAMPLES = int(SR * HOP_SEC)         # 80000

# 1. 严格照抄原代码的 ONNX 载入方式 (Cell 4)
def find_onnx():
    for p in INPUT_ROOT.rglob("perch_v2.onnx"):
        return p
    raise FileNotFoundError("找不到 perch_v2.onnx")

ONNX_PERCH_PATH = find_onnx()

_so = ort.SessionOptions()
_so.intra_op_num_threads = 4  # 原代码指定的线程数

# 如果用 GPU 加速提特征，自动检测并启用 CUDA
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if "CUDAExecutionProvider" in ort.get_available_providers() else ["CPUExecutionProvider"]

ONNX_SESSION = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so, providers=providers)
ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
ONNX_OUT_MAP = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
print(f"✅ ONNX 模型加载成功！使用的后端: {ONNX_SESSION.get_providers()[0]}")


print("环境配置严格执行完毕！")

ONNX Runtime installed
✅ TF 2.20 installed from wheel
✅ ONNX 模型加载成功！使用的后端: CPUExecutionProvider
环境配置严格执行完毕！


In [3]:
#cell2 %% [code]
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import os
import joblib
from tqdm.auto import tqdm
import gc

# ================= 配置区 =================
# 1. 训练模式设定："range" (区间训练) 或 "list" (指定下标训练)
TRAIN_MODE = "range" 

# 模式为 "range" 时生效 (闭区间)
TRAIN_RANGE =[0, 233]

# 模式为 "list" 时生效 (输入你想要单独训练的下标列表)
TRAIN_LIST = [15, 26, 73, 8]

# 2. 保存路径配置 (模型存入子文件夹，生成统一的报告)
SAVE_DIR = "/kaggle/working/saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)
REPORT_CSV = "/kaggle/working/training_report.csv"

# 3. PCA 配置
PCA_DIM = 1024
SAVE_PCA_NAME = f"emb_pca_{PCA_DIM}.npy"

# 4. 训练与模型配置
MAX_EPOCHS = 300
PATIENCE = 20
LEARNING_RATE = 1e-3
BATCH_SIZE = 128       
TEMP_T = 2.0           
NOISE_STD = 0.01       

# 5. 数据路径
FILE1_PATH = "/kaggle/input/notebooks/waozhang/bc2026-perch-label1/extracted_features/teacher_features_label.parquet"
FILE2_PATH = "/kaggle/input/notebooks/waozhang/bc2026-perch-label1/extracted_features/teacher_mixup_features_label.parquet"

# 💥 新增：无标签的真实环境音频特征 (用于打造抗噪 PCA)
UNLABELED_FILE_PATH = "/kaggle/input/notebooks/waozhang/bc2026-perch-featrue1/extracted_features/train_soundscapes_features.parquet"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用的计算设备: {DEVICE}")
print(f"模型保存目录: {SAVE_DIR}")

当前使用的计算设备: cpu
模型保存目录: /kaggle/working/saved_models


In [4]:
#cell3 %% [code]
print("开始加载两个 Parquet 文件...")
df1 = pd.read_parquet(FILE1_PATH, columns=['primary_label', 'embedding'])
df2 = pd.read_parquet(FILE2_PATH, columns=['primary_label', 'embedding'])

# 强行合并
df_all = pd.concat([df1, df2], ignore_index=True)
print(f"合并完成，总特征行数: {len(df_all)}")

# 1. 严格按照你要求的方法进行标签计数校验
print("\n=== 开始执行你要求的物种计数校验 ===")
m = df_all['primary_label'].values
a = {}
for i in m:
    if i is None or pd.isna(i): continue
    for j in i.split(';'):
        j = j.strip()
        if not j: continue
        if j not in a:
            a[j] = 1
        else:
            a[j] += 1

print(f"数据集中共发现 {len(a)} 种标签。")
# 打印前20个让你对一下本地数据
print("部分物种出现次数展示 (前20个):")
for k, v in list(a.items())[:20]:
    print(f"  {k}: {v}")

# 2. 获取目标物种
ALL_SPECIES = sorted(list(a.keys()))

# 3. 构建多标签 One-Hot 矩阵
print("\n正在构建多标签 One-Hot 矩阵...")
Y_matrix = np.zeros((len(df_all), len(ALL_SPECIES)), dtype=np.int8)
species_to_idx = {sp: i for i, sp in enumerate(ALL_SPECIES)}

for row_idx, labels_str in enumerate(tqdm(m, desc="构建标签")):
    if labels_str is None or pd.isna(labels_str): continue
    for sp in labels_str.split(';'):
        sp = sp.strip()
        if sp in species_to_idx:
            Y_matrix[row_idx, species_to_idx[sp]] = 1

# 4. 根据正样本数量正序排序，生成训练队列
counts = Y_matrix.sum(axis=0)
sorted_order = np.argsort(counts) # 正序索引
TRAIN_QUEUE =[(i, sorted_order[i], ALL_SPECIES[sorted_order[i]], counts[sorted_order[i]]) for i in range(len(ALL_SPECIES))]

print(f"\n物种排序完毕！最少的正样本数为: {TRAIN_QUEUE[0][3]}，最多的为: {TRAIN_QUEUE[-1][3]}")

开始加载两个 Parquet 文件...
合并完成，总特征行数: 55616

=== 开始执行你要求的物种计数校验 ===
数据集中共发现 234 种标签。
部分物种出现次数展示 (前20个):
  1161364: 263
  116570: 234
  1176823: 267
  1595929: 253
  209233: 213
  22930: 261
  22956: 284
  22961: 333
  22967: 569
  22973: 690
  22983: 265
  22985: 251
  23150: 212
  23154: 258
  23158: 625
  23176: 254
  23724: 208
  24279: 641
  24285: 271
  24287: 266

正在构建多标签 One-Hot 矩阵...


构建标签:   0%|          | 0/55616 [00:00<?, ?it/s]


物种排序完毕！最少的正样本数为: 6，最多的为: 944


In [5]:
#cell4 %% [code]
if os.path.exists(SAVE_PCA_NAME) and os.path.exists("global_scaler.pkl") and os.path.exists(f"global_pca_{PCA_DIM}.pkl"):
    print(f"检测到本地已有 {SAVE_PCA_NAME} 及拟合好的 PCA/Scaler 模型，直接加载加速！")
    X_pca = np.load(SAVE_PCA_NAME)
else:
    print("\n=== 开始全局 PCA 降维 (混合有标签 + 无标签野外环境数据) ===")
    
    # 提取之前 Cell 3 加载好的有标签特征 (这部分最终会被送去训练)
    X_raw = np.stack(df_all['embedding'].values)
    
    print("1. 正在读取 12 万行无标签野外环境特征...")
    df_unlabeled = pd.read_parquet(UNLABELED_FILE_PATH, columns=['embedding'])
    X_unlabeled = np.stack(df_unlabeled['embedding'].values)
    
    # 及时释放巨大的 dataframe 内存
    del df_unlabeled
    gc.collect()
    
    print(f"2. 拼接数据用于拟合尺子... 有标签: {len(X_raw)} 行, 无标签环境音: {len(X_unlabeled)} 行")
    X_combined = np.concatenate([X_raw, X_unlabeled], axis=0)
    
    # 释放无标签数组的单独内存
    del X_unlabeled
    gc.collect()
    
    print("3. 正在拟合 StandardScaler (混合数据)...")
    scaler = StandardScaler()
    scaler.fit(X_combined) # 让 Scaler 学习野外环境的均值和方差
    
    # 为了内存安全，原地转换
    X_combined_scaled = scaler.transform(X_combined)
    del X_combined
    gc.collect()
    
    print(f"4. 正在进行全局 PCA 拟合 (目标维度: {PCA_DIM}, 使用 randomized 求解器加速)...")
    # 使用 randomized 可以极大地加快几十万行矩阵的 SVD 求解速度，同时降低内存峰值
    pca = PCA(n_components=PCA_DIM, svd_solver='randomized', random_state=42)
    pca.fit(X_combined_scaled) # 让 PCA 学习包含野外底噪的通用坐标系
    
    explained_var = pca.explained_variance_ratio_.sum()
    print(f"✅ 全局 PCA 拟合完毕！降到 {PCA_DIM} 维，保留了原始混合数据 {explained_var*100:.2f}% 的信息量！")
    
    # 彻底释放用来拟合的巨大混合矩阵
    del X_combined_scaled
    gc.collect()
    
    print("5. 正在提取并转换有标签训练集的 PCA 特征 (完美对齐标签行数)...")
    # 注意：我们只把有标签的 X_raw 倒进打磨好的尺子里
    X_raw_scaled = scaler.transform(X_raw)
    X_pca = pca.transform(X_raw_scaled).astype(np.float32)
    
    # 释放原始特征内存
    del X_raw, X_raw_scaled
    gc.collect()
    
    # 保存通用的尺子和特征
    np.save(SAVE_PCA_NAME, X_pca)
    joblib.dump(scaler, "global_scaler.pkl")
    joblib.dump(pca, f"global_pca_{PCA_DIM}.pkl")
    print(f"✅ 已成功输出训练用对齐特征文件: {SAVE_PCA_NAME}")

# 释放 Cell 3 一开始读取的大表内存
try:
    del df_all
except NameError:
    pass
gc.collect()


=== 开始全局 PCA 降维 (混合有标签 + 无标签野外环境数据) ===
1. 正在读取 12 万行无标签野外环境特征...
2. 拼接数据用于拟合尺子... 有标签: 55616 行, 无标签环境音: 127896 行
3. 正在拟合 StandardScaler (混合数据)...
4. 正在进行全局 PCA 拟合 (目标维度: 1024, 使用 randomized 求解器加速)...
✅ 全局 PCA 拟合完毕！降到 1024 维，保留了原始混合数据 98.22% 的信息量！
5. 正在提取并转换有标签训练集的 PCA 特征 (完美对齐标签行数)...
✅ 已成功输出训练用对齐特征文件: emb_pca_1024.npy


0

In [6]:
#cell5 %% [code]
class SpeciesMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256,64),
            nn.ReLU(),
            nn.Linear(64, 1) # 输出原始 Logits
        )
        
    def forward(self, x):
        return self.net(x).squeeze(-1)

def fill_to_target(base_array, target_size):
    """
    智能扩充函数：优先保证整数倍数平铺，不够的再随机无放回抽取，完美契合要求。
    """
    n_base = len(base_array)
    if n_base == 0: return np.array([], dtype=int)
    repeats = target_size // n_base
    remainder = target_size % n_base
    
    res = np.tile(base_array, repeats)
    if remainder > 0:
        res = np.concatenate([res, np.random.choice(base_array, remainder, replace=False)])
    return res

In [7]:
# %% [code]
import os

# --- 解析训练模式 ---
if TRAIN_MODE == "range":
    start_idx, end_idx = TRAIN_RANGE
    end_idx = min(end_idx, len(TRAIN_QUEUE) - 1)
    tasks = TRAIN_QUEUE[start_idx:end_idx+1]
    print(f"\n🚀 当前模式 [Range]: 训练下标 {start_idx} 到 {end_idx} (共 {len(tasks)} 个物种)")
elif TRAIN_MODE == "list":
    tasks = [TRAIN_QUEUE[i] for i in TRAIN_LIST if i < len(TRAIN_QUEUE)]
    print(f"\n🚀 当前模式 [List]: 训练指定下标 {TRAIN_LIST} (共 {len(tasks)} 个物种)")
else:
    raise ValueError("TRAIN_MODE 必须是 'range' 或 'list'")

report_data =[]  # 用于收集 CSV 报告数据

for queue_idx, sp_idx, sp_name, pos_count in tasks:
    print(f"\n{'='*70}")
    print(f"🔥 正在训练[下标 {queue_idx}/233]: {sp_name} | 总正样本: {pos_count}")
    
    all_pos_idx = np.where(Y_matrix[:, sp_idx] == 1)[0]
    all_neg_idx = np.where(Y_matrix[:, sp_idx] == 0)[0]
    
    best_overall_auc = 0.0
    best_overall_state = None
    best_X_val = None
    best_y_val = None
    
    if pos_count < 40:
        print(">> 归类：[<40样本]，启动 5折 交叉验证 (保留全局验证集最高 AUC 的模型)")
        np.random.shuffle(all_pos_idx)
        folds_pos = np.array_split(all_pos_idx, 5)
        
        for fold in range(5):
            val_pos_idx = folds_pos[fold]
            train_pos_base = np.setdiff1d(all_pos_idx, val_pos_idx)
            
            if len(val_pos_idx) == 0: continue
                
            val_neg_count = len(val_pos_idx) * 20
            np.random.shuffle(all_neg_idx)
            val_neg_idx = all_neg_idx[:val_neg_count]
            train_neg_pool = all_neg_idx[val_neg_count:]
            
            target_pos_train = min(len(train_pos_base) * 4, 100)
            train_pos_aug_idx = fill_to_target(train_pos_base, target_pos_train)
            target_neg_train = len(train_pos_aug_idx) * 9
            
            model = SpeciesMLP(PCA_DIM).to(DEVICE)
            optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
            
            X_val = torch.tensor(np.concatenate([X_pca[val_pos_idx], X_pca[val_neg_idx]]), dtype=torch.float32).to(DEVICE)
            y_val = torch.tensor(np.concatenate([np.ones(len(val_pos_idx)), np.zeros(len(val_neg_idx))]), dtype=torch.float32).to(DEVICE)
            
            best_fold_auc = 0.0
            patience_counter = 0
            
            print(f"\n  >>>[Fold {fold+1}/5] 验证集(正:{len(val_pos_idx)}, 负:{len(val_neg_idx)}) | 训练集/Epoch(正:{target_pos_train}, 负:{target_neg_train})")
            
            for epoch in range(1, MAX_EPOCHS + 1):
                model.train()
                curr_neg_train_idx = np.random.choice(train_neg_pool, target_neg_train, replace=False)
                
                X_train_pos_np = X_pca[train_pos_aug_idx].copy()
                X_train_pos_np += np.random.normal(0, NOISE_STD, size=X_train_pos_np.shape).astype(np.float32)
                X_train_neg_np = X_pca[curr_neg_train_idx]
                
                X_train = torch.tensor(np.concatenate([X_train_pos_np, X_train_neg_np]), dtype=torch.float32)
                y_train = torch.tensor(np.concatenate([np.ones(len(X_train_pos_np)), np.zeros(len(X_train_neg_np))]), dtype=torch.float32)
                
                dataset = TensorDataset(X_train, y_train)
                loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
                
                n_pos_ep = y_train.sum()
                n_total_ep = y_train.numel()
                dyn_weight = torch.clamp((n_total_ep - n_pos_ep) / (n_pos_ep + 1), max=25.0).to(DEVICE)
                criterion = nn.BCEWithLogitsLoss(pos_weight=dyn_weight)
                
                epoch_loss = 0.0
                for batch_X, batch_y in loader:
                    batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                    optimizer.zero_grad()
                    logits = model(batch_X)
                    loss = criterion(logits / TEMP_T, batch_y)
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item() * len(batch_y)
                epoch_loss /= len(dataset)
                
                model.eval()
                with torch.no_grad():
                    val_logits = model(X_val)
                    val_probs = torch.sigmoid(val_logits / TEMP_T).cpu().numpy()
                    try:
                        auc = roc_auc_score(y_val.cpu().numpy(), val_probs)
                    except ValueError:
                        auc = 0.0
                
                if auc > best_fold_auc:
                    best_fold_auc = auc
                    patience_counter = 0
                    if auc > best_overall_auc:
                        best_overall_auc = auc
                        best_overall_state = model.state_dict().copy()
                        best_X_val = X_val.clone()
                        best_y_val = y_val.clone()
                else:
                    patience_counter += 1
                    
                if epoch % 10 == 0 or epoch == 1 or patience_counter >= PATIENCE:
                    print(f"      Epoch [{epoch:3d}/{MAX_EPOCHS}] | Train Loss: {epoch_loss:.4f} | 惩罚倍数: {dyn_weight.item():.2f} | Val AUC: {auc:.4f} | 早停计数: {patience_counter}/{PATIENCE} | 最佳整体AUC: {best_overall_auc:.4f}")
                    
                if patience_counter >= PATIENCE:
                    print(f"      [Fold {fold+1}] 触发早停。当前折最佳 AUC: {best_fold_auc:.4f}")
                    break
                    
    elif 40 <= pos_count < 240:
        print(">> 归类：[40~240样本]，不交叉验证，跑1个模型")
        val_pos_count = int(pos_count * 0.2)
        val_neg_count = val_pos_count * 5
        
        np.random.shuffle(all_pos_idx)
        val_pos_idx = all_pos_idx[:val_pos_count]
        train_pos_base = all_pos_idx[val_pos_count:]
        
        np.random.shuffle(all_neg_idx)
        val_neg_idx = all_neg_idx[:val_neg_count]
        train_neg_pool = all_neg_idx[val_neg_count:]
        
        target_pos_train = 200
        target_neg_train = 800
        train_pos_aug_idx = fill_to_target(train_pos_base, target_pos_train)
        
        print(f"\n  >>> 验证集(正:{len(val_pos_idx)}, 负:{len(val_neg_idx)}) | 训练集/Epoch(正:{target_pos_train}, 负:{target_neg_train})")
        
        model = SpeciesMLP(PCA_DIM).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        X_val = torch.tensor(np.concatenate([X_pca[val_pos_idx], X_pca[val_neg_idx]]), dtype=torch.float32).to(DEVICE)
        y_val = torch.tensor(np.concatenate([np.ones(len(val_pos_idx)), np.zeros(len(val_neg_idx))]), dtype=torch.float32).to(DEVICE)
        
        patience_counter = 0
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            curr_neg_train_idx = np.random.choice(train_neg_pool, target_neg_train, replace=False)
            
            X_train_pos_np = X_pca[train_pos_aug_idx].copy()
            X_train_pos_np += np.random.normal(0, NOISE_STD, size=X_train_pos_np.shape).astype(np.float32)
            X_train_neg_np = X_pca[curr_neg_train_idx]
            
            X_train = torch.tensor(np.concatenate([X_train_pos_np, X_train_neg_np]), dtype=torch.float32)
            y_train = torch.tensor(np.concatenate([np.ones(len(X_train_pos_np)), np.zeros(len(X_train_neg_np))]), dtype=torch.float32)
            
            dataset = TensorDataset(X_train, y_train)
            loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
            
            n_pos_ep = y_train.sum()
            n_total_ep = y_train.numel()
            dyn_weight = torch.clamp((n_total_ep - n_pos_ep) / (n_pos_ep + 1), max=25.0).to(DEVICE)
            criterion = nn.BCEWithLogitsLoss(pos_weight=dyn_weight)
            
            epoch_loss = 0.0
            for batch_X, batch_y in loader:
                batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                optimizer.zero_grad()
                logits = model(batch_X)
                loss = criterion(logits / TEMP_T, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item() * len(batch_y)
            epoch_loss /= len(dataset)
            
            model.eval()
            with torch.no_grad():
                val_logits = model(X_val)
                val_probs = torch.sigmoid(val_logits / TEMP_T).cpu().numpy()
                try:
                    auc = roc_auc_score(y_val.cpu().numpy(), val_probs)
                except ValueError:
                    auc = 0.0
            
            if auc > best_overall_auc:
                best_overall_auc = auc
                best_overall_state = model.state_dict().copy()
                patience_counter = 0
                best_X_val = X_val.clone()
                best_y_val = y_val.clone()
            else:
                patience_counter += 1
                
            if epoch % 10 == 0 or epoch == 1 or patience_counter >= PATIENCE:
                print(f"      Epoch[{epoch:3d}/{MAX_EPOCHS}] | Train Loss: {epoch_loss:.4f} | 惩罚倍数: {dyn_weight.item():.2f} | Val AUC: {auc:.4f} | 早停计数: {patience_counter}/{PATIENCE} | 最佳整体AUC: {best_overall_auc:.4f}")
                
            if patience_counter >= PATIENCE:
                print(f"      触发早停。当前最佳 AUC: {best_overall_auc:.4f}")
                break
                
    else: 
        print(">> 归类：[>=240样本]，不交叉验证，正负样本均动态抽取")
        val_pos_count = 40
        val_neg_count = 200
        
        np.random.shuffle(all_pos_idx)
        val_pos_idx = all_pos_idx[:val_pos_count]
        train_pos_pool = all_pos_idx[val_pos_count:]
        
        np.random.shuffle(all_neg_idx)
        val_neg_idx = all_neg_idx[:val_neg_count]
        train_neg_pool = all_neg_idx[val_neg_count:]
        
        target_pos_train = 200
        target_neg_train = 800
        
        print(f"\n  >>> 验证集(正:{len(val_pos_idx)}, 负:{len(val_neg_idx)}) | 训练集/Epoch(正:{target_pos_train}, 负:{target_neg_train})")
        
        model = SpeciesMLP(PCA_DIM).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        X_val = torch.tensor(np.concatenate([X_pca[val_pos_idx], X_pca[val_neg_idx]]), dtype=torch.float32).to(DEVICE)
        y_val = torch.tensor(np.concatenate([np.ones(len(val_pos_idx)), np.zeros(len(val_neg_idx))]), dtype=torch.float32).to(DEVICE)
        
        patience_counter = 0
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            curr_pos_train_idx = np.random.choice(train_pos_pool, target_pos_train, replace=False)
            curr_neg_train_idx = np.random.choice(train_neg_pool, target_neg_train, replace=False)
            
            X_train_pos_np = X_pca[curr_pos_train_idx].copy()
            X_train_pos_np += np.random.normal(0, NOISE_STD, size=X_train_pos_np.shape).astype(np.float32)
            X_train_neg_np = X_pca[curr_neg_train_idx]
            
            X_train = torch.tensor(np.concatenate([X_train_pos_np, X_train_neg_np]), dtype=torch.float32)
            y_train = torch.tensor(np.concatenate([np.ones(len(X_train_pos_np)), np.zeros(len(X_train_neg_np))]), dtype=torch.float32)
            
            dataset = TensorDataset(X_train, y_train)
            loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
            
            n_pos_ep = y_train.sum()
            n_total_ep = y_train.numel()
            dyn_weight = torch.clamp((n_total_ep - n_pos_ep) / (n_pos_ep + 1), max=25.0).to(DEVICE)
            criterion = nn.BCEWithLogitsLoss(pos_weight=dyn_weight)
            
            epoch_loss = 0.0
            for batch_X, batch_y in loader:
                batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                optimizer.zero_grad()
                logits = model(batch_X)
                loss = criterion(logits / TEMP_T, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item() * len(batch_y)
            epoch_loss /= len(dataset)
            
            model.eval()
            with torch.no_grad():
                val_logits = model(X_val)
                val_probs = torch.sigmoid(val_logits / TEMP_T).cpu().numpy()
                try:
                    auc = roc_auc_score(y_val.cpu().numpy(), val_probs)
                except ValueError:
                    auc = 0.0
            
            if auc > best_overall_auc:
                best_overall_auc = auc
                best_overall_state = model.state_dict().copy()
                patience_counter = 0
                best_X_val = X_val.clone()
                best_y_val = y_val.clone()
            else:
                patience_counter += 1
                
            if epoch % 10 == 0 or epoch == 1 or patience_counter >= PATIENCE:
                print(f"      Epoch[{epoch:3d}/{MAX_EPOCHS}] | Train Loss: {epoch_loss:.4f} | 惩罚倍数: {dyn_weight.item():.2f} | Val AUC: {auc:.4f} | 早停计数: {patience_counter}/{PATIENCE} | 最佳整体AUC: {best_overall_auc:.4f}")
                
            if patience_counter >= PATIENCE:
                print(f"      触发早停。当前最佳 AUC: {best_overall_auc:.4f}")
                break

    # ==========================================
    # 推理最佳模型，统计、计算 100% Precision 阈值并生成报告
    # ==========================================
    if best_overall_state is not None and best_X_val is not None:
        eval_model = SpeciesMLP(PCA_DIM).to(DEVICE)
        eval_model.load_state_dict(best_overall_state)
        eval_model.eval()
        
        with torch.no_grad():
            raw_logits = eval_model(best_X_val).cpu().numpy()
            true_labels = best_y_val.cpu().numpy()
            
        pos_logits = raw_logits[true_labels == 1]
        neg_logits = raw_logits[true_labels == 0]
        
        pos_mean = pos_logits.mean() if len(pos_logits) > 0 else np.nan
        pos_std  = pos_logits.std()  if len(pos_logits) > 0 else np.nan
        pos_min  = pos_logits.min()  if len(pos_logits) > 0 else np.nan
        pos_max  = pos_logits.max()  if len(pos_logits) > 0 else np.nan
        
        neg_mean = neg_logits.mean() if len(neg_logits) > 0 else np.nan
        neg_std  = neg_logits.std()  if len(neg_logits) > 0 else np.nan
        neg_min  = neg_logits.min()  if len(neg_logits) > 0 else np.nan
        neg_max  = neg_logits.max()  if len(neg_logits) > 0 else np.nan
        
        # 💥 新增逻辑：计算 100% 信任阈值 (即严格大于所有负样本的最大 Logits) 及符合该条件的阳性样本比例
        if len(neg_logits) > 0 and len(pos_logits) > 0:
            trust_logit = neg_max
            trust_count = (pos_logits > trust_logit).sum()
            trust_ratio = trust_count / len(pos_logits)
        else:
            trust_logit = np.nan
            trust_ratio = np.nan
        
        print(f"\n📊 --- 最佳验证集真实 Logits 统计分析 (未除以 T，未 Sigmoid) ---")
        if len(pos_logits) > 0:
            print(f"  [正样本 (N={len(pos_logits)})] 均值: {pos_mean:.4f} | 标准差: {pos_std:.4f} | Min: {pos_min:.4f} | Max: {pos_max:.4f}")
        else:
            print(f"  [正样本] 数量为 0，无法统计。")
            
        if len(neg_logits) > 0:
            print(f"  [负样本 (N={len(neg_logits)})] 均值: {neg_mean:.4f} | 标准差: {neg_std:.4f} | Min: {neg_min:.4f} | Max: {neg_max:.4f}")
        else:
            print(f"  [负样本] 数量为 0，无法统计。")
            
        if not np.isnan(trust_logit):
            print(f"  [置信指标] 100% 精确率阈值 (trust_logit): > {trust_logit:.4f} | 召回占比 (trust_ratio): {trust_ratio*100:.2f}% ({trust_count}/{len(pos_logits)})")
            
        # 保存模型至子文件夹
        safe_sp_name = sp_name.replace('/', '_').replace(' ', '_')
        save_name = f"1pmlp_{queue_idx}_{safe_sp_name}_{best_overall_auc:.4f}.pth"
        save_path = os.path.join(SAVE_DIR, save_name)
        torch.save(best_overall_state, save_path)
        print(f"🎉 训练完成！最高验证 AUC: {best_overall_auc:.4f} | 已保存至: {save_path}")
        
        # 记录到字典 (全英文列名)
        report_data.append({
            "index": queue_idx,
            "modelname": save_name,
            "pavg": pos_mean,
            "pstd": pos_std,
            "pmax": pos_max,
            "pmin": pos_min,
            "navg": neg_mean,
            "nstd": neg_std,
            "nmax": neg_max,
            "nmin": neg_min,
            "bestAUC": best_overall_auc,
            "trust_logit": trust_logit,
            "trust_ratio": trust_ratio
        })
    else:
        print(f"⚠️ 警告: 该物种未能产生有效的模型权重。")

# ==========================================
# 导出 CSV 报告
# ==========================================
if len(report_data) > 0:
    df_report = pd.DataFrame(report_data)
    
    # 列排序规范化 (严格遵循要求的全英文字段)
    cols_order = ['index', 'modelname', 'pavg', 'pstd', 'pmax', 'pmin', 'navg', 'nstd', 'nmax', 'nmin', 'bestAUC', 'trust_logit', 'trust_ratio']
    df_report = df_report[cols_order]
    
    # 如果文件已存在，则追加 (不写入表头)；否则新建
    if os.path.exists(REPORT_CSV):
        df_report.to_csv(REPORT_CSV, mode='a', header=False, index=False)
        print(f"\n📈 本次训练结果已成功追加至报告: {REPORT_CSV}")
    else:
        df_report.to_csv(REPORT_CSV, mode='w', header=True, index=False)
        print(f"\n📈 训练报告 CSV 已成功创建: {REPORT_CSV}")

print("\n✅ 当前设置的训练任务全部执行完毕！")


🚀 当前模式 [Range]: 训练下标 0 到 233 (共 234 个物种)

🔥 正在训练[下标 0/233]: 47158son05 | 总正样本: 6
>> 归类：[<40样本]，启动 5折 交叉验证 (保留全局验证集最高 AUC 的模型)

  >>>[Fold 1/5] 验证集(正:2, 负:40) | 训练集/Epoch(正:16, 负:144)
      Epoch [  1/300] | Train Loss: 1.1740 | 惩罚倍数: 8.47 | Val AUC: 1.0000 | 早停计数: 0/20 | 最佳整体AUC: 1.0000
      Epoch [ 10/300] | Train Loss: 0.0951 | 惩罚倍数: 8.47 | Val AUC: 1.0000 | 早停计数: 9/20 | 最佳整体AUC: 1.0000
      Epoch [ 20/300] | Train Loss: 0.0055 | 惩罚倍数: 8.47 | Val AUC: 1.0000 | 早停计数: 19/20 | 最佳整体AUC: 1.0000
      Epoch [ 21/300] | Train Loss: 0.0080 | 惩罚倍数: 8.47 | Val AUC: 1.0000 | 早停计数: 20/20 | 最佳整体AUC: 1.0000
      [Fold 1] 触发早停。当前折最佳 AUC: 1.0000

  >>>[Fold 2/5] 验证集(正:1, 负:20) | 训练集/Epoch(正:20, 负:180)
      Epoch [  1/300] | Train Loss: 1.2021 | 惩罚倍数: 8.57 | Val AUC: 1.0000 | 早停计数: 0/20 | 最佳整体AUC: 1.0000
      Epoch [ 10/300] | Train Loss: 0.1135 | 惩罚倍数: 8.57 | Val AUC: 1.0000 | 早停计数: 9/20 | 最佳整体AUC: 1.0000
      Epoch [ 20/300] | Train Loss: 0.0071 | 惩罚倍数: 8.57 | Val AUC: 1.0000 | 早停计数: 19/20 |